# Hangman strategy explorer

Compare the hangman strategies from `strategy.py`, `optimal.py` and `winprob.py`
on one word length under a chosen word distribution. Pick a `LENGTH`, a
`WEIGHTING` (how likely each word is to be the secret word) and a `STRATEGY`,
then re-run the rest: the opener, hardest words, per-guess tree, and how the
headline number scales with length all follow your choice.

Reminders: a *miss* is a wrong letter guess. `optimal`/`winprob` replay the
solvers' committed trees, so they only cover length/weighting combinations that
have already been solved -- others report cleanly instead of erroring.

In [ ]:
from pathlib import Path
import sys

def find_math_dir():
    cwd = Path.cwd().resolve()
    for c in (cwd, cwd.parent, cwd.parent.parent, cwd / 'math', cwd.parent / 'math'):
        if (c / 'strategy.py').exists():
            return c
    raise FileNotFoundError('Could not find strategy.py. Open from the repo root, math/, or math/notebooks/.')

math_dir = find_math_dir()
if str(math_dir) not in sys.path:
    sys.path.insert(0, str(math_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import strategy
import strategy_view as sview
import wordlist

print(f'imported strategy + strategy_view from {math_dir}')

## Pick length, distribution, and strategy

Change `LENGTH`, `WEIGHTING`, `STRATEGY` (and `BUDGET`, for winprob), then re-run
the rest.

`WEIGHTING` -- how likely each word is to be the secret word:
- `"uniform"` -- every word equally likely.
- `"zipf"` -- real English frequency (common words weigh more).
- `"greedy"` / `"best_opener"` -- the adversarial word-setter from `freq_search.py`
  (per-(letter,pattern) multipliers), tuned to make that guesser work hardest.
- `"greedy_word"` / `"best_opener_word"` -- the newer *per-word* frequency adversary.

`STRATEGY` -- the guesser:
- `"greedy"` -- guess the highest combined-frequency un-guessed letter (live, any weighting).
- `"greedy_best"` -- force the best first letter, then greedy (live, any weighting).
- `"optimal"` -- minimise expected misses (replays `data/shared/optimal/`).
- `"winprob"` -- maximise P(win within `BUDGET` misses) (replays `data/shared/winprob/`).

`make_strategy` returns one uniform `sv` that every cell below uses. `optimal`
and `winprob` exist only for solved length/weighting combos; when one is missing,
`sv.available` is `False` and `sv.note` says what to run -- the cells skip cleanly.

In [ ]:
LENGTH    = 5           # <-- word length
WEIGHTING = "uniform"   # <-- uniform, zipf, greedy, best_opener, greedy_word, best_opener_word
STRATEGY  = "greedy"    # <-- greedy, greedy_best, optimal, winprob
BUDGET    = None         # <-- winprob only: miss budget k (None = most permissive solved)

g  = strategy.GreedyGame(LENGTH)
sv = sview.make_strategy(STRATEGY, LENGTH, weighting=WEIGHTING, budget=BUDGET, game=g)

print(f'{g.N:,} words of length {LENGTH}  (zipf >= {wordlist.DEFAULT_ZIPF})')
print(f'weighting: {WEIGHTING}   strategy: {sv.label}')
if not sv.available:
    print(f'\n!! not available: {sv.note}')
else:
    print(f'opener: {sv.opener()!r}')
    hv = sv.headline_value()
    print(f'{sv.headline_label()}: ' + (f'{hv:.4f}' if hv is not None else 'n/a'))
    for k, v in sv.summary().items():
        print(f'  {k}: {v}')
    if sv.note:
        print(f'note: {sv.note}')

## Best opening letters

Expected misses if each letter is forced as the first guess and the guesser plays
on; the dashed line marks the best (its own opener). This forced-opener sweep is
greedy-family only -- for `optimal`/`winprob` it would need a re-solve per opener
(not cached), so those print a short note instead.

In [ ]:
by_letter = sv.misses_by_starting_letter() if sv.available else None
if by_letter is None:
    print(f'{sv.label}: forced-opener sweep is greedy-family only.')
    if sv.available:
        print(f'(this strategy opens with {sv.opener()!r})')
else:
    mb = pd.Series(by_letter, name='avg_misses'); mb.index.name = 'first_letter'
    op = sv.opener()
    fig, ax = plt.subplots(figsize=(10, 3.5))
    ax.bar(mb.index, mb.values, color='#3b6ea5')
    ax.axhline(mb[op], color='#cc4444', ls='--', lw=1, label=f'{sv.label} opener {op!r}')
    ax.set_ylabel('avg misses')
    ax.set_title(f'Expected misses by opening letter (length {LENGTH}, {WEIGHTING})')
    ax.legend(); plt.show()
    display(mb.to_frame().T)

## Per-word difficulty

How many misses the strategy makes on each word, and the distribution. Words the
strategy leaves *unresolved* -- `optimal` words capped past the solved depth, or
`winprob` words that can't be won within the budget -- have no finite miss count
and are tallied separately (they sort to the top of the hardest list).

In [ ]:
if not sv.available:
    print('not available:', sv.note)
else:
    pwm = sv.per_word_misses()
    resolved = pwm[~np.isnan(pwm)]
    n_unres = int(np.isnan(pwm).sum())
    words = pd.DataFrame({'word': g.words, 'misses': pwm})

    print(f'solved with 0 misses: {int((resolved == 0).sum()):,} words')
    hi = int(resolved.max()) if resolved.size else 0
    print(f'max misses: {hi}   mean (resolved): {resolved.mean():.3f}')
    if n_unres:
        kind = 'capped' if sv.kind == 'optimal' else 'lost (over budget)'
        print(f'unresolved ({kind}): {n_unres:,} words')

    fig, ax = plt.subplots(figsize=(9, 3))
    if resolved.size:
        ax.hist(resolved, bins=range(0, hi + 2), color='#3b6ea5', align='left', rwidth=0.85)
    ax.set_xlabel('misses'); ax.set_ylabel('# words')
    ax.set_title(f'Misses per word (length {LENGTH}, {sv.label})')
    plt.show()

    print('hardest words:')
    display(words.sort_values('misses', ascending=False, na_position='first')
                 .head(12).reset_index(drop=True))

## Trace one word

The actual guess sequence for a word under the current strategy: each guess and
whether it hit. Defaults to the hardest resolved word; set `WORD` yourself to
trace another.

In [ ]:
if not sv.available:
    print('not available:', sv.note)
else:
    pwm = sv.per_word_misses()
    WORD = g.words[int(np.argsort(np.where(np.isnan(pwm), -1.0, pwm))[-1])]  # hardest resolved
    # WORD = 'bench'   # <-- or pick your own

    hits = sv.trace(WORD)
    n_miss = sum(1 for _, hit in hits if not hit)
    print(f'{WORD!r}: {n_miss} misses  ({sv.label})')
    for letter, hit in hits:
        print(f'  {letter}  ' + ('hit ' if hit else 'MISS'))
    if not hits:
        print('  (the strategy leaves this word unresolved)')

## Inspect a tree node

Give a board (revealed letters + `_` blanks, e.g. `"b_ll_"`) and the missed
letters; `show_state` finds that node in the current strategy's tree: the
candidates remaining, the next guess, and every outcome it can produce. States
the strategy never reaches are flagged. A hit reveals **all** positions of a
letter, so put every occurrence of a revealed letter on the board.

In [ ]:
def show_state(board, missed=""):
    """Locate `board` (+ missed letters) in the current strategy `sv`'s tree and
    print its next guess and every outcome that guess can lead to."""
    if not sv.available:
        print('not available:', sv.note); return None
    st = sv.state(board, missed)
    tag = 'reachable' if st.get('reachable') else 'NOT REACHED'
    print(f"board {st['board']!r}   missed [{','.join(st['missed'])}]   ({tag})  [{sv.label}]")
    if not st.get('reachable'):
        print(f"  {st.get('reason')}"); return st
    if st['determined']:
        w = st['candidates'][0] if st['candidates'] else '(none)'
        print(f"  word determined: {w}"); return st
    print(f"  {st['n']} candidates, e.g. {', '.join(st['candidates'][:8])}")
    print(f"  guesses: {st['guess']!r}")
    for o in st['outcomes']:
        kind = 'HIT ' if o['hit'] else 'miss'
        print(f"    {kind} -> {o['board']!r}  ({o['n']:>4} words)  e.g. {', '.join(o['sample'][:5])}")
    return st

# the opener: the outcomes of the strategy's first guess (empty board)
_ = show_state('_' * LENGTH)

## All situations at depth k

Every node the strategy can reach after exactly `k` guesses, with the choice it
makes there. Depth 0 is the opener; at depth `k`, `k` guesses are in and `guess`
is the next one. Branches already resolved by depth `k` show `guess = None`.

In [ ]:
DEPTH = 3      # <-- number of guesses already made

if not sv.available:
    print('not available:', sv.note); sit = pd.DataFrame()
else:
    sit = pd.DataFrame(sv.states_at_depth(DEPTH))
    if len(sit):
        sit['missed'] = sit['missed'].apply(''.join)
        sit['sample'] = sit['sample'].apply(lambda s: ' '.join(s[:4]))
        live = sit[~sit['determined']]
        print(f'depth {DEPTH}: {len(sit)} nodes  ({len(live)} still undetermined)  [{sv.label}]')
        if len(live):
            spread = ', '.join(f'{k}={v}' for k, v in live['guess'].value_counts().items())
            print(f'next guess spread: {spread}')
        display(sit.sort_values('n', ascending=False).head(50).reset_index(drop=True))
    else:
        print(f'no nodes at depth {DEPTH}')

In [ ]:
sit['guess'].value_counts() if len(sit) else 'no nodes at this depth'

## Headline metric by length

How the current strategy's headline number -- avg misses, or P(win<=k) for
winprob -- scales across word lengths under the chosen `WEIGHTING`. Lengths with
no committed `optimal`/`winprob` solve are left as gaps.

In [ ]:
lengths = list(range(1, 13))
vals = []
for L in lengths:
    s = sview.make_strategy(STRATEGY, L, weighting=WEIGHTING, budget=BUDGET)
    vals.append(s.headline_value() if s.available else np.nan)
metric = sv.headline_label() if sv.available else 'metric'

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(lengths, vals, 'o-', color='#3b6ea5')
ax.set_xlabel('word length'); ax.set_ylabel(metric)
ax.set_title(f'{STRATEGY}: {metric} by word length ({WEIGHTING})')
ax.grid(alpha=0.3); plt.show()

def _fmt(v):
    return None if v is None or (isinstance(v, float) and np.isnan(v)) else round(v, 3)
pd.DataFrame({'length': lengths, metric: [_fmt(v) for v in vals]})